In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Ajouter la racine du projet au PYTHONPATH
PROJECT_ROOT = Path().cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)

# Charger le .env
load_dotenv(PROJECT_ROOT / "config" / ".env")

print("ENV loaded from:", PROJECT_ROOT / "config" / ".env")


import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

from src.utils.s3_io import read_parquet_from_s3

Project root added: c:\Users\mirei\Desktop\BiblioTech
ENV loaded from: c:\Users\mirei\Desktop\BiblioTech\config\.env


## Chargement des données

In [2]:
books = read_parquet_from_s3("silver", "books_clean.parquet")
ratings = read_parquet_from_s3("silver", "ratings_joinable.parquet")
popularity = read_parquet_from_s3("gold", "book_popularity.parquet")

print(books.shape, ratings.shape, popularity.shape)

(271062, 8) (383851, 3) (149721, 11)


### Chargement des données pour l’analyse avancée

Les données utilisées dans cette analyse proviennent du Data Lake (MinIO), plus précisément :

- `books_clean` (silver) : 271 062 livres
- `ratings_joinable` (silver) : 383 851 interactions utilisateur-livre
- `book_popularity` (gold) : 149 721 livres enrichis


### Interprétation

On remarque que :

- le nombre de livres dans `book_popularity` est inférieur à `books_clean`
- cela s’explique par le fait que : seuls les livres ayant au moins un rating sont conservés dans la table gold


### Rôle de chaque dataset

- `books` → métadonnées (titre, auteur, année)
- `ratings` → interactions utilisateur
- `popularity` → dataset enrichi prêt pour l’analyse et la recommandation


### Conclusion

La table `book_popularity` constitue la base principale pour les analyses avancées :

- elle combine informations livres + agrégations
- elle est directement exploitable pour les visualisations et les modèles

## PARTIE 1 — VISUALISATION MÉTIER

#### Top livres les plus populaires

Quels livres sont les plus connus / les plus lus ?

In [3]:
top_books = popularity.sort_values("ratings_count", ascending=False).head(10)

fig = px.bar(
    top_books,
    x="ratings_count",
    y="title",
    orientation="h",
    title="Top 10 livres les plus populaires",
)

fig.show()

### Top livres les mieux notés (avec filtre)

Quels livres sont vraiment appréciés par les lecteurs ?

In [4]:
filtered = popularity[popularity["ratings_count"] >= 20]

top_rated = filtered.sort_values("average_rating", ascending=False).head(10)

fig = px.bar(
    top_rated,
    x="average_rating",
    y="title",
    orientation="h",
    title="Top livres les mieux notés (>= 20 ratings)",
)

fig.show()

### Différence entre popularité et qualité

Deux approches sont utilisées pour analyser les livres :

#### Popularité
Basée sur le nombre de ratings :
- mesure la visibilité d’un livre
- reflète son succès ou sa diffusion


#### Qualité (note moyenne)
Basée sur la moyenne des notes :
- mesure l’appréciation des lecteurs
- nécessite un nombre minimum de ratings pour être fiable


### Problème des faibles volumes

Un livre avec peu de ratings peut avoir une note élevée mais peu représentative.

Exemple :
- 1 rating → note = 10 → non fiable


### Solution : filtrage

On impose un seuil minimum (ex : 20 ratings) pour :
- garantir la robustesse des résultats
- éviter les biais statistiques


### Conclusion

- Popularité ≠ qualité
- Un bon système de recommandation doit combiner les deux

### Top auteurs

In [5]:
top_authors = (
    popularity.groupby("author")["ratings_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_authors,
    x="ratings_count",
    y="author",
    orientation="h",
    title="Top auteurs par popularité",
)

fig.show()

## PARTIE 2 — ANALYSE STATISTIQUE

### Corrélation popularité vs note

Pour comprendre le lien entre 

In [6]:
corr_pearson = pearsonr(
    popularity["ratings_count"],
    popularity["average_rating"]
)

corr_spearman = spearmanr(
    popularity["ratings_count"],
    popularity["average_rating"]
)

print("Pearson:", corr_pearson)
print("Spearman:", corr_spearman)

Pearson: PearsonRResult(statistic=np.float64(0.01980246043899413), pvalue=np.float64(1.8162323854509234e-14))
Spearman: SignificanceResult(statistic=np.float64(-0.010277078934828841), pvalue=np.float64(6.988881584921535e-05))


### Corrélation entre popularité et note moyenne

Deux types de corrélation ont été calculés :

- Pearson (relation linéaire)
- Spearman (relation monotone)

Résultats :

- Pearson ≈ 0.02
- Spearman ≈ -0.01


### Interprétation

Les coefficients sont très proches de 0, ce qui indique :

**absence de relation significative entre popularité et note moyenne**

En d’autres termes :
- un livre très populaire n’est pas forcément mieux noté
- un livre peu populaire peut être très bien noté


### À propos des p-values

Les p-values sont très faibles, ce qui signifie que :

- la corrélation est statistiquement significative
- MAIS l’effet est extrêmement faible

Cela s’explique par la taille importante du dataset


### Conclusion

La popularité d’un livre n’est pas un bon indicateur de sa qualité perçue.

Il est donc nécessaire de :
- ne pas se baser uniquement sur le nombre de ratings
- combiner plusieurs critères (ex : score pondéré)


### Implications pour la recommandation

- un système basé uniquement sur la popularité serait biaisé
- un système basé uniquement sur la note serait instable
- il faut combiner :
  - popularité
  - qualité
  - comportement utilisateur

### Distribution par année

In [7]:
year_stats = (
    popularity.groupby("year_of_publication")["ratings_count"]
    .sum()
    .reset_index()
)

fig = px.line(
    year_stats,
    x="year_of_publication",
    y="ratings_count",
    title="Évolution du nombre de ratings par année",
)

fig.show()

## PARTIE 3 — ANALYSE AVANCÉE

### Distribution du score pondéré

In [8]:
fig = px.histogram(
    popularity,
    x="weighted_score",
    nbins=50,
    title="Distribution du score pondéré",
)

fig.show()